In [ ]:
import pandas as pd
import numpy as np
import re
import sqlite3 as sql
import mysql.connector
import unicodedata
import html 

conn = mysql.connector.connect(
   host="db.example.com",
   user="fake_db_user",
   password="FAKE_DB_PASSWORD",
   database="fake_db_name",
   consume_results=True
 )
print(conn)

In [ ]:
df= pd.read_sql(""" 

SELECT DISTINCT
ot.ticket_id as "Id", ot.number as "Ticket Number",
ote.created as "Last Updated",cdta.subject as "Subject", 
ote.body as "Description",ote.poster as "From",sta.name as "Status" , 
ote.type as "Type"
FROM ost_ticket AS ot, 
ost_thread AS th, 
ost_thread_entry AS ote , 
ost_ticket_status AS sta, 
ost_ticket__cdata AS cdta
WHERE ote.thread_id = th.id AND 
th.object_id = ot.ticket_id  AND 
ot.status_id = sta.id AND
ot.ticket_id  = cdta.ticket_id AND
sta.name='Closed'
ORDER BY ote.id desc 

""",conn)


In [ ]:
dictTicketRequest = {}
dictTicketMessage = {}
merged_dict = {}

for i in range(len(df)):
    ticketNo = df.iloc[i]["Ticket Number"]
    if df.iloc[i]["Type"] == "R" and "Agent One" not in df.iloc[i]["From"] and "Manager One" not in df.iloc[i]["From"]:
        if ticketNo not in dictTicketRequest: dictTicketRequest[ticketNo] = []
        dictTicketRequest[ticketNo].append(df.iloc[i])
        
    elif df.iloc[i]["Type"] == "M" and "Agent One" not in df.iloc[i]["From"] and "Manager One" not in df.iloc[i]["From"]:
        if ticketNo not in dictTicketMessage: dictTicketMessage[ticketNo] = []
        dictTicketMessage[ticketNo].append(df.iloc[i])

        

for ticketNo, records_R in dictTicketRequest.items():
    if ticketNo in dictTicketMessage:
        merged_dict[ticketNo] = {"R": records_R, "M": dictTicketMessage[ticketNo]}



In [ ]:
        
merged_dict.values() #Ticket karşılık gelen verileri görelim
len(merged_dict) # Veri setindeki uzunluğa bakalım
merged_dict.keys() #18792 kayıtta(M-R) oluşuyor


In [ ]:
merged_dict['032176']

In [ ]:
# merged_dict'i DataFrame'e dönüştürme
data = []
for Id, records in merged_dict.items():
    row = {'Id': Id}
    for record_type, record_list in records.items():
        for idx, record in enumerate(record_list):
            for col, value in record.items():
                row[col] = value
    data.append(row)

a = pd.DataFrame(data)

In [ ]:
a

In [ ]:
a["clean_description"] = a["Description"].apply(lambda x: x.lower())
a["clean_subject"] = a["Subject"].apply(lambda x: x.lower())
a.head()

In [ ]:
def remove_URL(Description): # description kısmındaki url temizliyoruz

    return re.sub(r"https?://\S+|www\.\S+", "", Description)
# Urls text den çıkarma
a["clean_description"] = a["clean_description"].apply(lambda x: remove_URL(x))
a["clean_description"][31]

In [ ]:
# Remove HTML tags (HTML etiketlerini kadlırma)
# Description ve Subject kısmı için

from bs4 import BeautifulSoup
import re
# same using beautifulsoup
# casefold = lower işlevi görür ingilizce dışındaki diller iç,n geçerlidir.
# beautfiul soup uygulayıp adım adım yeni bir kolon daha ekleyelim
def remove_html_escape(html):
    soup = BeautifulSoup(str(html).strip().casefold(), "lxml")
    text = soup.get_text(separator= " ") # div veya herhangi bir tag metinden çıkarken kelimlerin bitişik olmasını engellemek için " "  koyduk
    text = re.sub(r'[\s\r\n]+',' ',text)
    return text.strip()

data = pd.DataFrame(a)
data['clean_description_html'] = a['clean_description'].apply(remove_html_escape) 
data["clean_description_html"]

In [ ]:
def remove_greetings_from_text(text, greetings):
    text_lower = text.casefold()
    for greeting in greetings:
        text = text.replace(greeting, "")
    return text

greetings = [
    "selamlar","sayın yetkili","değerli acente2","merhabalar","merhaba",
    "i̇yi günler","iyi çalışmalar","rica etsem","arkadaşlar","bey" ,"görüşmemize istinaden",
    "kolay gelsin","değerli acente","i̇ş ortağımız","ailesi","+ destek@",
    "sn yetkili","dün telefonla görüşmemize istinaden","tekrardan","tekrar","sonsuz teşekkürler",
    "agent1","agent2","agent3","agent4","agent5","agent6","agent7","agent8","agent9",
    "akif","yasin","tugay","murat","furkan","melih","selim","yasemin","sena","hanım","hanim","bey",
    "bu ","veya ","yada ","ya da ","ve ","class=""msonormal"">"
   ]


# Her bir veri örneği için greetings kelimelerini temizleme işlemi
cleaned_data_set = []
for text in data["clean_description_html"]:
    cleaned_text = remove_greetings_from_text(text, greetings)
    cleaned_data_set.append(cleaned_text)

    
# Temizlediğimiz datayı DataFrame olarak bir sutün daha ekleyip yazıralım onun üzerinden işlemlerimize devam edelim
data['text_clean_greeting'] = cleaned_data_set

In [ ]:
data['text_clean_greeting'][133]

In [ ]:
def preprocess(text):
    text = re.sub(r'[^\w\s\']','', text) # noktalama işaretlerini kaldırmak için
    text = re.sub(r'\s best regards.*', ' ', text) # devamındaki kısımları silmek istersek
    text = re.sub(r'\s saygılarımla.*', ' ', text) # devamındaki kısımları silmek istersek
    text = re.sub(r'\s kimden:.*','',text)
    text = re.sub('\r\n', ' ', text) # \xa0 silmek istersek
    text = re.sub(r'\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Z|a-z]{2,}\b',' ',text) # eposta adresi
  #  text = re.sub(r'\skişisel verilerinizin.*',' ',text)
  #  text = re.sub(r'\sbu mesaj ve ekleri.*',' ',text)
  #  text = re.sub(r'gizlilik notu','',text)
    text = re.sub(r'p\b',' ',text) # metinde kalmış ise p tag alır
    text = re.sub(r'<br\s?\/?>',' ',text) # br tag kaldırma için
    text = re.sub(r'\S+@\S+','', text) # elif@gmail.com vb. ifadeleri kaldırır.
    
 # Telefon numaralarını kaldır # metin içindeki türk telefon numrası formatına ait sayıları siler
 # Bir sürü tel yazma formatı odluğu için bir kaç tane ekliyoruz
    text = re.sub(r'\b\d{10}\b|\b\d{7,12}\b', '', text) 
    text = re.sub(r'\b\d{4}\s\d{3}\s\d{4}\b','',text)
    text = re.sub(r'\b\d{4}\s\d{3}\s\d{2}\s\d{2}\b','',text)
    text = re.sub(r'\b\d{4}\s\d{3}\s\d{1}\s\d{3}\b','',text)
    text = re.sub(r'\b\d{2}\s\d{3}\s\d{3}\s\d{4}\b','',text)



    return text.strip().lower() 

data['text_clean_preprocess'] = data['text_clean_greeting'].map(preprocess)


In [ ]:
data['text_clean_preprocess'][133]

In [ ]:
data


In [ ]:
# Sadece belirli kolonların alt kümesini göre ver'iyi kaydededelim etiketsiz olarak
column = data[['Id', 'Ticket Number','Last Updated','Subject','Description','From','Status','Type','text_clean_preprocess']] 
column.to_csv('Genel.csv',index=False)